<a href="https://colab.research.google.com/github/springboardmentor443m-coder/AI-Skin-Intelligence-Personalized-Skincare-Planner/blob/Subhransu-Sekhar/Recommendation_pipeline_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files
# Main dataset
# Simple upload
uploaded = files.upload()

# Load the csv
raw_df = pd.read_csv('final_skincare_v13_complete.csv')
print(f"Dataset loaded successfully. Shape: {raw_df.shape}")
display(raw_df.head())

Saving final_skincare_v13_complete.csv to final_skincare_v13_complete.csv
Dataset loaded successfully. Shape: (2286, 11)


,product_id,brand_name,product_name,price,product_type,loves_count_norm,rating_norm,reviews_norm,skin_types,skin_concerns,ingredients_cleaned
0,P439055,Algenist,GENIUS Sleeping Collagen Moisturizer,98.0,Moisturizers,0.031360,0.90826,0.081958,"Combination, Dry, Normal","Collagen/Plumping, Firmness/Elasticity, Hypoal...","collagen (vegan) , water (aqua, eau), ethylhex..."
1,P421277,Algenist,GENIUS Liquid Collagen Serum,115.0,Treatments/Serums,0.062766,0.80518,0.071907,"Combination, Dry, Normal","Collagen/Plumping, Firmness/Elasticity, Hypoal...","collagen (vegan) , water (aqua, eau), propaned..."
2,P467602,Algenist,Triple Algae Eye Renewal Balm Eye Cream,68.0,Eye Care,0.016545,0.90612,0.070852,Universal,General Care,"aqua (water/eau), stearic acid, isopropyl isos..."
3,P432045,Algenist,GENIUS Liquid Collagen Lip Treatment,29.0,Lip Care,0.041106,0.77442,0.040266,"Combination, Dry, Normal","Collagen/Plumping, Firmness/Elasticity, Hypoal...","collagen (vegan) , water (aqua, eau), glycerin..."
4,P311143,Algenist,SUBLIME DEFENSE Ultra Lightweight UV Defense F...,28.0,Face Sunscreen,0.025227,0.88268,0.031518,"Combination, Dry, Normal","Hypoallergenic, Sun Protection","octinoxate 7.5%, titanium dioxide 2%, zinc oxi..."


In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

# =====================================================================
# 1. CONFIGURATION & ECOSYSTEM SETUP
# =====================================================================
df = pd.read_csv('final_skincare_v13_complete.csv')

SKIN_TYPES = ['Universal', 'Normal', 'Combination', 'Dry', 'Oily']
SKIN_CONCERNS = [
    'Hydration/Dryness', 'Dullness/Texture', 'Anti-Aging', 'General Care',
    'Firmness/Elasticity', 'Pores', 'Brightening', 'Dark Circles',
    'Acne/Blemishes', 'Dark Spots', 'Collagen/Plumping', 'Sun Protection',
    'Redness', 'Hypoallergenic'
]
FEATURE_SPACE = SKIN_TYPES + SKIN_CONCERNS

# =====================================================================
# 2. ADVANCED SANITIZATION LAYER (Calibrated to CSV Anomalies)
# =====================================================================
def isolate_clinical_actives(ingredient_string):
    if pd.isna(ingredient_string) or str(ingredient_string).strip() == '':
        return ""

    raw_ingredients = str(ingredient_string).split(',')
    valid_actives = []

    # UPDATED: Added fragrance allergens, base emollients, and color codes found in your CSV
    filler_raw_string = (
        r'water|aqua|eau|extract|oil|seed|leaf|root|bark|flower|'
        r'juice|stem|powder|butter|wax|cera|'
        r'glycol|glycerin|propanediol|butylene|hexanediol|methylpropanediol|'
        r'alcohol|phenoxyethanol|paraben|chlorphenesin|silicone|dimethicone|'
        r'benzoate|sorbate|hydroxide|chloride|sulfate|'
        r'edta|gum|crosspolymer|copolymer|carbomer|'
        r'polysorbate|steareth|stearate|cetyl|cetearyl|'
        r'peg-\d+|ppg-\d+|poloxamer|'
        r'parfum|fragrance|aroma|flavor|limonene|linalool|geraniol|citronellol|citral|'
        r'ethylhexylglycerin|caprylic capric triglyceride|bht|hydroxyacetophenone|'
        r'mica|silica|glycine|talc|alumina|tin oxide|'
        r'ci\s?\d+|lake|iron oxides|red \d+|blue \d+|yellow \d+'
    )
    filler_pattern = re.compile(rf'\b({filler_raw_string})\b', re.IGNORECASE)

    for ing in raw_ingredients:
        clean_ing = ing.strip().lower()

        # DROP: Instructional text and product names (e.g., "step 1:", "pekee bar:")
        if clean_ing.endswith(':'):
            continue

        # DROP: Standalone numbers or single characters (e.g., "1", "2")
        if clean_ing.isnumeric() or len(clean_ing) <= 2:
            continue

        # CLEAN: Strip percentages to normalize dosage (e.g., "zinc oxide 10%" -> "zinc oxide")
        clean_ing = re.sub(r'\d+(\.\d+)?\s*%', '', clean_ing)

        # Normalize internal formatting noise
        normalized_ing = re.sub(r'[\/\(\)\.]', ' ', clean_ing).strip()
        normalized_ing = re.sub(r'\s+', ' ', normalized_ing)

        # Explicit drop conditions for empty entries after cleaning
        if not normalized_ing or normalized_ing in ['substance', 'ingredients', 'nan']:
            continue

        # If the isolated ingredient passes the word boundary check, preserve it
        if not filler_pattern.search(normalized_ing):
            valid_actives.append(normalized_ing)

    return ", ".join(valid_actives)

print("⚙️ Sanitizing product corpus... (Dropping instructions, percentages, and fragrance)")
df['true_actives_only'] = df['ingredients_cleaned'].apply(isolate_clinical_actives)

# =====================================================================
# 3. CSV-CALIBRATED CLINICAL TRANSLATION MAP
# =====================================================================
# UPDATED: Mapped to the exact frequency hits in your provided dataset
CLINICAL_MAP_EXACT = {
    'Hydration/Dryness': ['hyaluronic acid', 'sodium hyaluronate', 'ceramide np', 'ceramide ap', 'ceramide eop', 'squalane', 'panthenol', 'polyglutamic acid', 'beta-glucan', 'sodium pca', 'trehalose', 'betaine', 'sodium acetylated hyaluronate'],
    'Dullness/Texture': ['lactic acid', 'glycolic acid', 'mandelic acid', 'gluconolactone', 'salicylic acid', 'papain', 'bromelain', 'malic acid', 'tartaric acid'],
    'Anti-Aging': ['retinol', 'retinal', 'bakuchiol', 'palmitoyl tetrapeptide-7', 'palmitoyl tripeptide-1', 'acetyl hexapeptide-8', 'ascorbic acid', 'resveratrol', 'adenosine', 'ubiquinone'],
    'General Care': ['allantoin', 'bisabolol', 'centella asiatica', 'tocopherol', 'tocopheryl acetate', 'niacinamide'],
    'Firmness/Elasticity': ['collagen', 'hydrolyzed collagen', 'acetyl hexapeptide-8', 'palmitoyl tripeptide-1', 'copper tripeptide-1', 'elastin', 'retinol'],
    'Pores': ['salicylic acid', 'niacinamide', 'charcoal', 'kaolin', 'bentonite', 'witch hazel'],
    'Brightening': ['ascorbic acid', 'tetrahexyldecyl ascorbate', 'ascorbyl glucoside', '3-o-ethyl ascorbic acid', 'niacinamide', 'glycyrrhiza glabra', 'licorice', 'alpha-arbutin', 'tranexamic acid', 'ferulic acid'],
    'Dark Circles': ['caffeine', 'ascorbic acid', 'retinol', 'niacinamide', 'phytonadione', 'hesperidin methyl chalcone'],
    'Acne/Blemishes': ['salicylic acid', 'benzoyl peroxide', 'sulfur', 'zinc pca', 'zinc gluconate', 'melaleuca alternifolia', 'tea tree'],
    'Dark Spots': ['tranexamic acid', 'kojic acid', 'alpha-arbutin', 'niacinamide', 'ascorbic acid', 'azelaic acid', 'glycolic acid'],
    'Collagen/Plumping': ['collagen', 'hydrolyzed collagen', 'hyaluronic acid', 'sodium hyaluronate', 'palmitoyl tripeptide-1', 'palmitoyl tetrapeptide-7', 'arginine', 'proline', 'serine'],
    'Sun Protection': ['zinc oxide', 'titanium dioxide', 'avobenzone', 'homosalate', 'octocrylene', 'octisalate'],
    'Redness': ['centella asiatica', 'madecassoside', 'azelaic acid', 'allantoin', 'bisabolol', 'panthenol', 'colloidal oatmeal'],
    'Hypoallergenic': ['colloidal oatmeal', 'allantoin', 'bisabolol', 'ceramide np', 'squalane']
}

# =====================================================================
# 4. REFERENCE MATRIX GENERATION
# =====================================================================
print("⚙️ Assembling structural profile reference matrices...")
product_profile_matrix = np.zeros((len(df), len(FEATURE_SPACE)))

for i in range(len(df)):
    row_types = str(df.iloc[i]['skin_types']).lower()
    row_concerns = str(df.iloc[i]['skin_concerns']).lower()

    for j, feature in enumerate(FEATURE_SPACE):
        feature_lower = feature.lower()
        if feature_lower in [s.lower() for s in SKIN_TYPES] and 'universal' in row_types:
            product_profile_matrix[i][j] = 1
        elif feature_lower in row_types or feature_lower in row_concerns:
            product_profile_matrix[i][j] = 1

print("⚙️ Fitting tokenized active chemistry TF-IDF vector space...")
tfidf_clean = TfidfVectorizer(min_df=2, max_df=0.70, ngram_range=(1, 3))
tfidf_clean_matrix = tfidf_clean.fit_transform(df['true_actives_only'])

# =====================================================================
# 5. ORCHESTRATION PIPELINE
# =====================================================================
def get_personalized_recommendations_with_alternatives(skin_type, concerns_list, budget, top_n=5):
    if isinstance(concerns_list, str):
        concerns_list = [concerns_list]

    # --- Phase A: Hard Filter Candidate Pool ---
    pool = df[df['price'] <= budget].copy()

    def verify_skin_safety(row_skin):
        row_skin_clean = str(row_skin).strip().lower()
        if skin_type.lower() in row_skin_clean or 'universal' in row_skin_clean:
            return True
        return False

    pool['is_safe'] = pool['skin_types'].apply(verify_skin_safety)
    pool = pool[pool['is_safe'] == True].drop(columns=['is_safe'])

    if pool.empty:
        print("❌ No dermatologically compatible products found within this budget profile.")
        return

    pool_indices = pool.index.tolist()

    # --- Phase B: User Profile Matrix Scoring ---
    user_profile_vector = np.zeros(len(FEATURE_SPACE))
    matched_skin = [s for s in SKIN_TYPES if s.lower() == skin_type.lower()]
    if matched_skin:
        user_profile_vector[FEATURE_SPACE.index(matched_skin[0])] = 1
    for concern in concerns_list:
        if concern in FEATURE_SPACE:
            user_profile_vector[FEATURE_SPACE.index(concern)] = 1

    sliced_profile_matrix = product_profile_matrix[pool_indices]
    pool['profile_score'] = cosine_similarity([user_profile_vector], sliced_profile_matrix).flatten()

    # --- Phase C: Pure Active Chemical Vector Scoring ---
    query_keywords = []
    for concern in concerns_list:
        query_keywords.extend(CLINICAL_MAP_EXACT.get(concern, []))
    compiled_query_text = " ".join(list(set(query_keywords)))

    user_chemical_vector = tfidf_clean.transform([compiled_query_text])
    sliced_tfidf_matrix = tfidf_clean_matrix[pool_indices]
    pool['ingredient_score'] = cosine_similarity(user_chemical_vector, sliced_tfidf_matrix).flatten()

    # --- Phase D: Pool-Wide Normalization & Synthesis ---
    scaler = MinMaxScaler()
    if len(pool) > 1:
        pool['scaled_profile'] = scaler.fit_transform(pool[['profile_score']])
        pool['scaled_ingredient'] = scaler.fit_transform(pool[['ingredient_score']])
    else:
        pool['scaled_profile'] = pool['profile_score']
        pool['scaled_ingredient'] = pool['ingredient_score']

    # 60% Active Chemistry Weight / 40% Categorical Intent Weight
    pool['suitability_score'] = (pool['scaled_ingredient'] * 0.6) + (pool['scaled_profile'] * 0.4)

    # Sort the global candidate pool by suitability and popularity metrics
    scored_pool = pool.sort_values(by=['suitability_score', 'loves_count_norm'], ascending=[False, False])

    # Isolate the definitive top 5 recommendations
    top_recommendations = scored_pool.head(top_n)
    top_indices = top_recommendations.index.tolist()

    # =====================================================================
    # 6. UNIFIED SYSTEM OUTPUT DISPLAY
    # =====================================================================
    print("\n" + "="*90)
    print(f"🌟 USER PROFILE: Skin Type: {skin_type} | Concerns: {concerns_list} | Budget Cap: ${budget}")
    print("="*90)

    print("\n👑 STEP 1: TOP 5 PERSONALIZED CLINICAL RECOMMENDATIONS")
    display(top_recommendations[[
        'brand_name', 'product_name', 'product_type', 'price',
        'skin_types', 'skin_concerns', 'profile_score', 'ingredient_score', 'suitability_score'
    ]])

    print("\n🔄 STEP 2: CATEGORY ALTERNATIVES UNDER FIXED USER BUDGET")

    # Iterate through the top 5 to generate independent sub-pool alternative lookups
    for rank, (idx, row) in enumerate(top_recommendations.iterrows(), 1):
        target_type = row['product_type']

        # Filter the global scored pool for matching category, under user budget, excluding primary recommended items
        alternatives_pool = scored_pool[
            (scored_pool['product_type'] == target_type) &
            (~scored_pool.index.isin(top_indices))
        ]

        # Extract top 3 alternatives matching the user's initial query vectors
        top_alternatives = alternatives_pool.head(3)

        print("-" * 90)
        suffix = "st" if rank == 1 else "nd" if rank == 2 else "rd" if rank == 3 else "th"
        print(f"✨ These 3 are the alternatives of the {rank}{suffix} of top 5 product:")
        print(f"   ↳ Parent Product: {row['brand_name']} - {row['product_name']} (${row['price']}) [{target_type}]{row ['skin_concerns']}  [{row ['skin_types']}]")
        print("-" * 90)

        if not top_alternatives.empty:
            display(top_alternatives[[
                'brand_name', 'product_name', 'product_type', 'price', 'skin_types', 'skin_concerns',
                'profile_score', 'ingredient_score', 'suitability_score'

            ]])
        else:
            print("   ℹ️ No additional alternative products available in this category under the budget constraint.")

    print("\n" + "="*90)

# =====================================================================
# 7. EXECUTION
# =====================================================================
get_personalized_recommendations_with_alternatives(
    skin_type="Dry",
    concerns_list=["Collagen/Plumping"],
    budget=70.00,
    top_n=5
)

⚙️ Sanitizing product corpus... (Dropping instructions, percentages, and fragrance)
⚙️ Assembling structural profile reference matrices...
⚙️ Fitting tokenized active chemistry TF-IDF vector space...

🌟 USER PROFILE: Skin Type: Dry | Concerns: ['Collagen/Plumping'] | Budget Cap: $70.0

👑 STEP 1: TOP 5 PERSONALIZED CLINICAL RECOMMENDATIONS


,brand_name,product_name,product_type,price,skin_types,skin_concerns,profile_score,ingredient_score,suitability_score
2104,The INKEY List,Hyaluronic Acid Hydrating Serum,Treatments/Serums,9.99,Universal,"Collagen/Plumping, Dark Circles, Hydration/Dry...",0.500000,0.378263,0.828660
2253,Wishful,Thirst Trap Juice Hyaluronic Acid & Peptide Hy...,Treatments/Serums,47.00,Universal,"Collagen/Plumping, Firmness/Elasticity, Hydrat...",0.500000,0.295993,0.698163
742,First Aid Beauty,Bounce-Boosting Serum with Collagen + Peptides,Treatments/Serums,44.00,"Combination, Dry, Normal","Anti-Aging, Collagen/Plumping",0.632456,0.200987,0.657046
1605,Peter Thomas Roth,FIRMx Collagen Face & Eye Hydra-Gel Patches,Eye Care,65.00,Universal,"Anti-Aging, Collagen/Plumping, Firmness/Elasti...",0.471405,0.277582,0.645303
947,HUM Nutrition,Collagen Love Skin Firming Supplement with Hya...,Wellness,40.00,Universal,"Anti-Aging, Brightening, Collagen/Plumping, Fi...",0.447214,0.289972,0.644942



🔄 STEP 2: CATEGORY ALTERNATIVES UNDER FIXED USER BUDGET
------------------------------------------------------------------------------------------
✨ These 3 are the alternatives of the 1st of top 5 product:
   ↳ Parent Product: The INKEY List - Hyaluronic Acid Hydrating Serum ($9.99) [Treatments/Serums]Collagen/Plumping, Dark Circles, Hydration/Dryness  [Universal]
------------------------------------------------------------------------------------------


,brand_name,product_name,product_type,price,skin_types,skin_concerns,profile_score,ingredient_score,suitability_score
2116,The INKEY List,Collagen Booster Firming Peptide Serum,Treatments/Serums,12.99,Universal,"Anti-Aging, Collagen/Plumping, Firmness/Elasti...",0.500000,0.241121,0.611126
2272,Youth To The People,Triple Peptide Hydrating + Firming Oasis Serum...,Treatments/Serums,54.00,"Combination, Dry, Normal","Collagen/Plumping, Hydration/Dryness",0.632456,0.155613,0.585074
1312,Mario Badescu,Vitamin C Serum,Treatments/Serums,45.00,Universal,Brightening,0.288675,0.254008,0.456738


------------------------------------------------------------------------------------------
✨ These 3 are the alternatives of the 2nd of top 5 product:
   ↳ Parent Product: Wishful - Thirst Trap Juice Hyaluronic Acid & Peptide Hydrating Facial Serum ($47.0) [Treatments/Serums]Collagen/Plumping, Firmness/Elasticity, Hydration/Dryness  [Universal]
------------------------------------------------------------------------------------------


...

,brand_name,product_name,product_type,price,skin_types,skin_concerns,profile_score,ingredient_score,suitability_score
2116,The INKEY List,Collagen Booster Firming Peptide Serum,Treatments/Serums,12.99,Universal,"Anti-Aging, Collagen/Plumping, Firmness/Elasti...",0.500000,0.241121,0.611126
2272,Youth To The People,Triple Peptide Hydrating + Firming Oasis Serum...,Treatments/Serums,54.00,"Combination, Dry, Normal","Collagen/Plumping, Hydration/Dryness",0.632456,0.155613,0.585074
1312,Mario Badescu,Vitamin C Serum,Treatments/Serums,45.00,Universal,Brightening,0.288675,0.254008,0.456738


------------------------------------------------------------------------------------------
✨ These 3 are the alternatives of the 4th of top 5 product:
   ↳ Parent Product: Peter Thomas Roth - FIRMx Collagen Face & Eye Hydra-Gel Patches ($65.0) [Eye Care]Anti-Aging, Collagen/Plumping, Firmness/Elasticity, Hydration/Dryness  [Universal]
------------------------------------------------------------------------------------------


,brand_name,product_name,product_type,price,skin_types,skin_concerns,profile_score,ingredient_score,suitability_score
66,BeautyBio,Bright Eyes Collagen-Infused Brightening Collo...,Eye Care,40.0,Universal,"Collagen/Plumping, Dark Circles, Dullness/Text...",0.471405,0.151213,0.444857
2238,Wander Beauty,Baggage Claim Eye Masks,Eye Care,26.0,Universal,"Collagen/Plumping, Hydration/Dryness",0.534522,0.066114,0.362091
2239,Wander Beauty,Baggage Claim Rose Gold Eye Masks,Eye Care,26.0,Universal,"Collagen/Plumping, Hydration/Dryness",0.534522,0.066114,0.362091


------------------------------------------------------------------------------------------
✨ These 3 are the alternatives of the 5th of top 5 product:
   ↳ Parent Product: HUM Nutrition - Collagen Love Skin Firming Supplement with Hyaluronic Acid & Vitamin C ($40.0) [Wellness]Anti-Aging, Brightening, Collagen/Plumping, Firmness/Elasticity, Hydration/Dryness  [Universal]
------------------------------------------------------------------------------------------


,brand_name,product_name,product_type,price,skin_types,skin_concerns,profile_score,ingredient_score,suitability_score
943,HUM Nutrition,Collagen POP + Vitamin C Dissolvable Tablets,Wellness,12.0,Universal,"Brightening, Collagen/Plumping, Firmness/Elast...",0.471405,0.095832,0.357013
2145,The Nue Co.,Skin Hydrator Ceramide and Hyaluronic Acid Sup...,Wellness,45.0,Universal,"Anti-Aging, Collagen/Plumping, Firmness/Elasti...",0.471405,0.064265,0.306940
1382,Murad,Pure Skin Clarifying Dietary Supplement,Wellness,50.0,Universal,General Care,0.288675,0.016060,0.079305


In [ ]:
import pandas as pd

# Split the comma-separated ingredients into individual rows
actives_series = (
    df['true_actives_only']
    .fillna('')
    .str.split(',')
    .explode()
    .str.strip()
    .str.lower()
)

# Remove empty values
actives_series = actives_series[
    (actives_series != '') &
    (actives_series.notna())
]

# Count occurrences
actives_inventory = (
    actives_series
    .value_counts()
    .rename_axis('Clinical Active')
    .reset_index(name='Count')
)

print(f"Total Unique Clinical Actives: {len(actives_inventory)}")

display(actives_inventory)

# Save CSV
actives_inventory.to_csv(
    "clinical_actives_inventory.csv",
    index=False
)

print("Saved as clinical_actives_inventory.csv")

Total Unique Clinical Actives: 2673


,Clinical Active,Count
0,tocopherol,1092
1,citric acid,1049
2,sodium hyaluronate,940
3,tocopheryl acetate,649
4,squalane,586
...,...,...
2668,hydroxycinnamic acid,1
2669,ethyl ferulate trehalose,1
2670,palmitoyl tetrapeptide-10,1
2671,sodium hydroxypropylphosphate,1


Saved as clinical_actives_inventory.csv


In [ ]:
import pandas as pd

# ============================
# Terms to audit
# ============================
check_terms = [
    'ppg',
    'glycol',
    'glycerin',
    'water',
    'aqua',
    'water (aqua, eau)',
    'collagen (vegan)',
    'oryza sativa (rice) bran extract'
]

# ==========================================================
# PART 1 : Audit clinical_actives_inventory_with_products.csv
# ==========================================================

inventory_df = pd.read_csv("clinical_actives_inventory.csv")

# Standardize
inventory_df['Clinical Active'] = (
    inventory_df['Clinical Active']
    .fillna('')
    .astype(str)
    .str.strip()
    .str.lower()
)

print("="*70)
print("AUDIT : clinical_actives_inventory.csv")
print("="*70)

csv_results = {}

for term in check_terms:

    matches = inventory_df[
        inventory_df['Clinical Active'] == term.lower()
    ]

    csv_results[term] = len(matches) > 0

    if len(matches):

        print(f"\n✅ FOUND : {term}")

        display(matches[['Clinical Active','Count']])

    else:

        print(f"\n❌ NOT FOUND : {term}")


# ==========================================================
# PART 2 : Audit df['true_actives_only']
# ==========================================================

actives_series = (
    df['true_actives_only']
      .fillna('')
      .str.split(',')
      .explode()
      .astype(str)
      .str.strip()
      .str.lower()
)

print("\n")
print("="*70)
print("AUDIT : df['true_actives_only']")
print("="*70)

df_results = {}

for term in check_terms:

    matches = actives_series[
        actives_series == term.lower()
    ]

    df_results[term] = len(matches) > 0

    if len(matches):

        print(f"\n✅ FOUND : {term}")
        print(f"Occurrences : {len(matches)}")

    else:

        print(f"\n❌ NOT FOUND : {term}")


# ==========================================================
# PART 3 : Final Comparison
# ==========================================================

print("\n")
print("="*70)
print("FINAL COMPARISON")
print("="*70)

comparison = pd.DataFrame({
    "Term": check_terms,
    "CSV": [csv_results[t] for t in check_terms],
    "DataFrame": [df_results[t] for t in check_terms]
})

comparison["Status"] = comparison["CSV"] == comparison["DataFrame"]

display(comparison)

if comparison["Status"].all():
    print("\n🎉 SUCCESS : CSV and DataFrame MATCH PERFECTLY.")
else:
    print("\n⚠ WARNING : Differences found.")

AUDIT : clinical_actives_inventory.csv

❌ NOT FOUND : ppg

❌ NOT FOUND : glycol

❌ NOT FOUND : glycerin

❌ NOT FOUND : water

❌ NOT FOUND : aqua

❌ NOT FOUND : water (aqua, eau)

❌ NOT FOUND : collagen (vegan)

❌ NOT FOUND : oryza sativa (rice) bran extract


AUDIT : df['true_actives_only']

❌ NOT FOUND : ppg

❌ NOT FOUND : glycol

❌ NOT FOUND : glycerin

❌ NOT FOUND : water

❌ NOT FOUND : aqua

❌ NOT FOUND : water (aqua, eau)

❌ NOT FOUND : collagen (vegan)

❌ NOT FOUND : oryza sativa (rice) bran extract


FINAL COMPARISON


,Term,CSV,DataFrame,Status
0,ppg,False,False,True
1,glycol,False,False,True
2,glycerin,False,False,True
3,water,False,False,True
4,aqua,False,False,True
5,"water (aqua, eau)",False,False,True
6,collagen (vegan),False,False,True
7,oryza sativa (rice) bran extract,False,False,True



🎉 SUCCESS : CSV and DataFrame MATCH PERFECTLY.


In [ ]:
#NLP+LIGHTBGM
import pandas as pd
import numpy as np
import re
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# 1. CONFIGURATION & ECOSYSTEM SETUP
# =====================================================================
df = pd.read_csv('final_skincare_v13_complete.csv')

SKIN_TYPES = ['Universal', 'Normal', 'Combination', 'Dry', 'Oily']
SKIN_CONCERNS = [
    'Hydration/Dryness', 'Dullness/Texture', 'Anti-Aging', 'General Care',
    'Firmness/Elasticity', 'Pores', 'Brightening', 'Dark Circles',
    'Acne/Blemishes', 'Dark Spots', 'Collagen/Plumping', 'Sun Protection',
    'Redness', 'Hypoallergenic'
]
FEATURE_SPACE = SKIN_TYPES + SKIN_CONCERNS

# =====================================================================
# 2. ADVANCED SANITIZATION LAYER
# =====================================================================
def isolate_clinical_actives(ingredient_string):
    if pd.isna(ingredient_string) or str(ingredient_string).strip() == '':
        return ""

    raw_ingredients = str(ingredient_string).split(',')
    valid_actives = []

    filler_raw_string = (
        r'water|aqua|eau|extract|oil|seed|leaf|root|bark|flower|'
        r'juice|stem|powder|butter|wax|cera|'
        r'glycol|glycerin|propanediol|butylene|hexanediol|methylpropanediol|'
        r'alcohol|phenoxyethanol|paraben|chlorphenesin|silicone|dimethicone|'
        r'benzoate|sorbate|hydroxide|chloride|sulfate|'
        r'edta|gum|crosspolymer|copolymer|carbomer|'
        r'polysorbate|steareth|stearate|cetyl|cetearyl|'
        r'peg-\d+|ppg-\d+|poloxamer|'
        r'parfum|fragrance|aroma|flavor|limonene|linalool|geraniol|citronellol|citral|'
        r'ethylhexylglycerin|caprylic capric triglyceride|bht|hydroxyacetophenone|'
        r'mica|silica|glycine|talc|alumina|tin oxide|'
        r'ci\s?\d+|lake|iron oxides|red \d+|blue \d+|yellow \d+'
    )
    filler_pattern = re.compile(rf'\b({filler_raw_string})\b', re.IGNORECASE)

    for ing in raw_ingredients:
        clean_ing = ing.strip().lower()
        if clean_ing.endswith(':'): continue
        if clean_ing.isnumeric() or len(clean_ing) <= 2: continue
        clean_ing = re.sub(r'\d+(\.\d+)?\s*%', '', clean_ing)
        normalized_ing = re.sub(r'[\/\(\)\.]', ' ', clean_ing).strip()
        normalized_ing = re.sub(r'\s+', ' ', normalized_ing)

        if not normalized_ing or normalized_ing in ['substance', 'ingredients', 'nan']:
            continue

        if not filler_pattern.search(normalized_ing):
            valid_actives.append(normalized_ing)

    return ", ".join(valid_actives)

print("⚙️ Sanitizing product corpus...")
df['true_actives_only'] = df['ingredients_cleaned'].apply(isolate_clinical_actives)

# =====================================================================
# 3. CSV-CALIBRATED CLINICAL TRANSLATION MAP
# =====================================================================
CLINICAL_MAP_EXACT = {
    'Hydration/Dryness': ['hyaluronic acid', 'sodium hyaluronate', 'ceramide np', 'ceramide ap', 'ceramide eop', 'squalane', 'panthenol', 'polyglutamic acid', 'beta-glucan', 'sodium pca', 'trehalose', 'betaine'],
    'Dullness/Texture': ['lactic acid', 'glycolic acid', 'mandelic acid', 'gluconolactone', 'salicylic acid', 'papain', 'bromelain', 'malic acid', 'tartaric acid'],
    'Anti-Aging': ['retinol', 'retinal', 'bakuchiol', 'palmitoyl tetrapeptide-7', 'palmitoyl tripeptide-1', 'acetyl hexapeptide-8', 'ascorbic acid', 'resveratrol', 'adenosine', 'ubiquinone'],
    'General Care': ['allantoin', 'bisabolol', 'centella asiatica', 'tocopherol', 'tocopheryl acetate', 'niacinamide'],
    'Firmness/Elasticity': ['collagen', 'hydrolyzed collagen', 'acetyl hexapeptide-8', 'palmitoyl tripeptide-1', 'copper tripeptide-1', 'elastin', 'retinol'],
    'Pores': ['salicylic acid', 'niacinamide', 'charcoal', 'kaolin', 'bentonite', 'witch hazel'],
    'Brightening': ['ascorbic acid', 'tetrahexyldecyl ascorbate', 'ascorbyl glucoside', '3-o-ethyl ascorbic acid', 'niacinamide', 'glycyrrhiza glabra', 'licorice', 'alpha-arbutin', 'tranexamic acid', 'ferulic acid'],
    'Dark Circles': ['caffeine', 'ascorbic acid', 'retinol', 'niacinamide', 'phytonadione', 'hesperidin methyl chalcone'],
    'Acne/Blemishes': ['salicylic acid', 'benzoyl peroxide', 'sulfur', 'zinc pca', 'zinc gluconate', 'melaleuca alternifolia', 'tea tree'],
    'Dark Spots': ['tranexamic acid', 'kojic acid', 'alpha-arbutin', 'niacinamide', 'ascorbic acid', 'azelaic acid', 'glycolic acid'],
    'Collagen/Plumping': ['collagen', 'hydrolyzed collagen', 'hyaluronic acid', 'sodium hyaluronate', 'palmitoyl tripeptide-1', 'palmitoyl tetrapeptide-7', 'arginine', 'proline', 'serine'],
    'Sun Protection': ['zinc oxide', 'titanium dioxide', 'avobenzone', 'homosalate', 'octocrylene', 'octisalate'],
    'Redness': ['centella asiatica', 'madecassoside', 'azelaic acid', 'allantoin', 'bisabolol', 'panthenol', 'colloidal oatmeal'],
    'Hypoallergenic': ['colloidal oatmeal', 'allantoin', 'bisabolol', 'ceramide np', 'squalane']
}

# =====================================================================
# 4. REFERENCE MATRIX GENERATION
# =====================================================================
print("⚙️ Assembling structural profile matrices & TF-IDF vector space...")
product_profile_matrix = np.zeros((len(df), len(FEATURE_SPACE)))

for i in range(len(df)):
    row_types = str(df.iloc[i]['skin_types']).lower()
    row_concerns = str(df.iloc[i]['skin_concerns']).lower()
    for j, feature in enumerate(FEATURE_SPACE):
        feature_lower = feature.lower()
        if feature_lower in [s.lower() for s in SKIN_TYPES] and 'universal' in row_types:
            product_profile_matrix[i][j] = 1
        elif feature_lower in row_types or feature_lower in row_concerns:
            product_profile_matrix[i][j] = 1

tfidf_clean = TfidfVectorizer(min_df=2, max_df=0.70, ngram_range=(1, 3))
tfidf_clean_matrix = tfidf_clean.fit_transform(df['true_actives_only'])

# =====================================================================
# 5. OFFLINE LIGHTGBM MODEL TRAINING
# =====================================================================
print("🧠 Training LightGBM Ranking Model...")
# Feature Engineering for the model
df['num_actives'] = df['true_actives_only'].apply(lambda x: len(str(x).split(',')) if x else 0)
df['product_type_cat'] = df['product_type'].astype('category')

lgbm_features = ['price', 'num_actives', 'product_type_cat']
lgbm_target = 'loves_count_norm'

# Initialize and train the Regressor to predict market engagement
lgbm_ranker = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
lgbm_ranker.fit(df[lgbm_features], df[lgbm_target])

# =====================================================================
# 6. ENTERPRISE TWO-STAGE INFERENCE PIPELINE
# =====================================================================
def two_stage_recommender(skin_type, concerns_list, budget, final_n=5, retrieval_n=15):
    if isinstance(concerns_list, str): concerns_list = [concerns_list]

    # --- STAGE 1: CANDIDATE RETRIEVAL (TF-IDF) ---
    pool = df[df['price'] <= budget].copy()

    def verify_skin_safety(row_skin):
        row_skin_clean = str(row_skin).strip().lower()
        return skin_type.lower() in row_skin_clean or 'universal' in row_skin_clean

    pool = pool[pool['skin_types'].apply(verify_skin_safety)]
    if pool.empty:
        print("❌ No dermatologically compatible products found within this budget.")
        return pd.DataFrame(), pd.DataFrame()

    pool_indices = pool.index.tolist()

    # Profile Scoring
    user_profile_vector = np.zeros(len(FEATURE_SPACE))
    matched_skin = [s for s in SKIN_TYPES if s.lower() == skin_type.lower()]
    if matched_skin: user_profile_vector[FEATURE_SPACE.index(matched_skin[0])] = 1
    for concern in concerns_list:
        if concern in FEATURE_SPACE: user_profile_vector[FEATURE_SPACE.index(concern)] = 1

    pool['profile_score'] = cosine_similarity([user_profile_vector], product_profile_matrix[pool_indices]).flatten()

    # Chemistry Scoring
    query_keywords = []
    for concern in concerns_list:
        query_keywords.extend(CLINICAL_MAP_EXACT.get(concern, []))
    compiled_query_text = " ".join(list(set(query_keywords)))

    user_chemical_vector = tfidf_clean.transform([compiled_query_text])
    pool['ingredient_score'] = cosine_similarity(user_chemical_vector, tfidf_clean_matrix[pool_indices]).flatten()

    # Base Suitability Score
    scaler = MinMaxScaler()
    pool['scaled_profile'] = scaler.fit_transform(pool[['profile_score']])
    pool['scaled_ingredient'] = scaler.fit_transform(pool[['ingredient_score']])
    pool['base_suitability_score'] = (pool['scaled_ingredient'] * 0.6) + (pool['scaled_profile'] * 0.4)

    # Extract Top 15 Candidates
    top_15_pool = pool.sort_values(by='base_suitability_score', ascending=False).head(retrieval_n).copy()

    # --- STAGE 2: LIGHTGBM RANKING (ML Layer) ---
    # Model predicts viability/engagement
    top_15_pool['lgbm_prediction'] = lgbm_ranker.predict(top_15_pool[lgbm_features])

    # Scale LGBM predictions to match the 0-1 range of the suitability score
    top_15_pool['scaled_lgbm'] = scaler.fit_transform(top_15_pool[['lgbm_prediction']])

    # Final Blend: 75% Clinical Accuracy / 25% Machine Learning Viability
    top_15_pool['final_two_stage_score'] = (top_15_pool['base_suitability_score'] * 0.75) + (top_15_pool['scaled_lgbm'] * 0.25)

    # Final Sort to get Top 5
    final_top_5 = top_15_pool.sort_values(by='final_two_stage_score', ascending=False).head(final_n)

    return final_top_5, pool


# =====================================================================
# 7. CORRECTED ORCHESTRATOR & ALTERNATIVE FINDER
# =====================================================================
def execute_system(skin_type, concerns_list, budget):
    final_top_5, global_pool = two_stage_recommender(skin_type, concerns_list, budget, final_n=5)

    if final_top_5.empty: return

    top_indices = final_top_5.index.tolist()

    print("\n" + "="*90)
    print(f"🌟 TWO-STAGE USER PROFILE: Skin Type: {skin_type} | Concerns: {concerns_list} | Budget: ${budget}")
    print("="*90)

    print("\n👑 STEP 1: TOP 5 ML-RANKED CLINICAL RECOMMENDATIONS")
    display(final_top_5[[
        'brand_name', 'product_name', 'product_type', 'price', 'skin_types', 'skin_concerns','profile_score', 'ingredient_score',
        'base_suitability_score', 'lgbm_prediction', 'final_two_stage_score'
    ]])

    print("\n🔄 STEP 2: CATEGORY ALTERNATIVES (FIXED USER BUDGET BENCH)")

    for rank, (idx, row) in enumerate(final_top_5.iterrows(), 1):
        target_type = row['product_type']

        # FIXED: Removed the parent price restriction.
        # Now searches the global pool up to the original user budget ceiling.
        alternatives_pool = global_pool[
            (global_pool['product_type'] == target_type) &
            (~global_pool.index.isin(top_indices))
        ].copy()

        suffix = "st" if rank == 1 else "nd" if rank == 2 else "rd" if rank == 3 else "th"
        print("-" * 90)
        print(f"✨ Top alternatives for the {rank}{suffix} product:")
        print(f"   ↳ Parent Product: {row['brand_name']} - {row['product_name']} (${row['price']}) [{target_type}]{row ['skin_concerns']}  [{row ['skin_types']}]")
        print("-" * 90)

        if not alternatives_pool.empty:
            # Score alternatives based on the user's query profile up to the fixed budget limit
            top_alternatives = alternatives_pool.sort_values(
                by=['base_suitability_score', 'loves_count_norm'], ascending=[False, False]
            ).head(3)

            display(top_alternatives[[
                'brand_name', 'product_name', 'product_type', 'price', 'skin_types', 'skin_concerns',
                'profile_score', 'ingredient_score','base_suitability_score'
            ]])
        else:
            print("   ℹ️ No additional alternative options found in this category under the budget limit.")

    print("\n" + "="*90)

# =====================================================================
# 8. LIVE INFERENCE TEST
# =====================================================================
execute_system(
    skin_type="Dry",
    concerns_list=["Collagen/Plumping"],
    budget=70.00
)

⚙️ Sanitizing product corpus...
⚙️ Assembling structural profile matrices & TF-IDF vector space...
🧠 Training LightGBM Ranking Model...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000084 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 230
[LightGBM] [Info] Number of data points in the train set: 2286, number of used features: 3
[LightGBM] [Info] Start training from score 0.025710

🌟 TWO-STAGE USER PROFILE: Skin Type: Dry | Concerns: ['Collagen/Plumping'] | Budget: $70.0

👑 STEP 1: TOP 5 ML-RANKED CLINICAL RECOMMENDATIONS


,brand_name,product_name,product_type,price,profile_score,ingredient_score,base_suitability_score,lgbm_prediction,final_two_stage_score
2104,The INKEY List,Hyaluronic Acid Hydrating Serum,Treatments/Serums,9.99,0.500000,0.378263,0.828660,0.175335,0.871495
2116,The INKEY List,Collagen Booster Firming Peptide Serum,Treatments/Serums,12.99,0.500000,0.241121,0.611126,0.098791,0.599642
2253,Wishful,Thirst Trap Juice Hyaluronic Acid & Peptide Hy...,Treatments/Serums,47.00,0.500000,0.295993,0.698163,0.027712,0.563980
742,First Aid Beauty,Bounce-Boosting Serum with Collagen + Peptides,Treatments/Serums,44.00,0.632456,0.200987,0.657046,0.022456,0.525676
947,HUM Nutrition,Collagen Love Skin Firming Supplement with Hya...,Wellness,40.00,0.447214,0.289972,0.644942,0.018672,0.511226



🔄 STEP 2: CATEGORY ALTERNATIVES (FIXED USER BUDGET BENCH)
------------------------------------------------------------------------------------------
✨ Top alternatives for the 1st product:
   ↳ Parent Product: The INKEY List - Hyaluronic Acid Hydrating Serum ($9.99) [Treatments/Serums]Collagen/Plumping, Dark Circles, Hydration/Dryness  [Universal]
------------------------------------------------------------------------------------------


,brand_name,product_name,product_type,price,skin_types,skin_concerns,profile_score,ingredient_score,base_suitability_score
2272,Youth To The People,Triple Peptide Hydrating + Firming Oasis Serum...,Treatments/Serums,54.0,"Combination, Dry, Normal","Collagen/Plumping, Hydration/Dryness",0.632456,0.155613,0.585074
1312,Mario Badescu,Vitamin C Serum,Treatments/Serums,45.0,Universal,Brightening,0.288675,0.254008,0.456738
2237,TULA Skincare,24-7 Ultra Hydration Triple-Hydra Complex Day ...,Treatments/Serums,48.0,Universal,"Collagen/Plumping, Hydration/Dryness",0.534522,0.119530,0.446819


------------------------------------------------------------------------------------------
✨ Top alternatives for the 2nd product:
   ↳ Parent Product: The INKEY List - Collagen Booster Firming Peptide Serum ($12.99) [Treatments/Serums]Anti-Aging, Collagen/Plumping, Firmness/Elasticity  [Universal]
------------------------------------------------------------------------------------------


...

,brand_name,product_name,product_type,price,skin_types,skin_concerns,profile_score,ingredient_score,base_suitability_score
2272,Youth To The People,Triple Peptide Hydrating + Firming Oasis Serum...,Treatments/Serums,54.0,"Combination, Dry, Normal","Collagen/Plumping, Hydration/Dryness",0.632456,0.155613,0.585074
1312,Mario Badescu,Vitamin C Serum,Treatments/Serums,45.0,Universal,Brightening,0.288675,0.254008,0.456738
2237,TULA Skincare,24-7 Ultra Hydration Triple-Hydra Complex Day ...,Treatments/Serums,48.0,Universal,"Collagen/Plumping, Hydration/Dryness",0.534522,0.119530,0.446819


------------------------------------------------------------------------------------------
✨ Top alternatives for the 4th product:
   ↳ Parent Product: First Aid Beauty - Bounce-Boosting Serum with Collagen + Peptides ($44.0) [Treatments/Serums]Anti-Aging, Collagen/Plumping  [Combination, Dry, Normal]
------------------------------------------------------------------------------------------


,brand_name,product_name,product_type,price,skin_types,skin_concerns,profile_score,ingredient_score,base_suitability_score
2272,Youth To The People,Triple Peptide Hydrating + Firming Oasis Serum...,Treatments/Serums,54.0,"Combination, Dry, Normal","Collagen/Plumping, Hydration/Dryness",0.632456,0.155613,0.585074
1312,Mario Badescu,Vitamin C Serum,Treatments/Serums,45.0,Universal,Brightening,0.288675,0.254008,0.456738
2237,TULA Skincare,24-7 Ultra Hydration Triple-Hydra Complex Day ...,Treatments/Serums,48.0,Universal,"Collagen/Plumping, Hydration/Dryness",0.534522,0.119530,0.446819


------------------------------------------------------------------------------------------
✨ Top alternatives for the 5th product:
   ↳ Parent Product: HUM Nutrition - Collagen Love Skin Firming Supplement with Hyaluronic Acid & Vitamin C ($40.0) [Wellness]Anti-Aging, Brightening, Collagen/Plumping, Firmness/Elasticity, Hydration/Dryness  [Universal]
------------------------------------------------------------------------------------------


,brand_name,product_name,product_type,price,skin_types,skin_concerns,profile_score,ingredient_score,base_suitability_score
943,HUM Nutrition,Collagen POP + Vitamin C Dissolvable Tablets,Wellness,12.0,Universal,"Brightening, Collagen/Plumping, Firmness/Elast...",0.471405,0.095832,0.357013
2145,The Nue Co.,Skin Hydrator Ceramide and Hyaluronic Acid Sup...,Wellness,45.0,Universal,"Anti-Aging, Collagen/Plumping, Firmness/Elasti...",0.471405,0.064265,0.306940
1382,Murad,Pure Skin Clarifying Dietary Supplement,Wellness,50.0,Universal,General Care,0.288675,0.016060,0.079305
